# AMAZTA: Adaptive Multi-Agent Zero Trust Architecture (End-to-End Implementation)

**Main dataset:** IEEE-CIS Fraud Detection (Kaggle competition)
**Case-study dataset:** Transaction Data for Banking Operations (Kaggle, `ziya07`)

## Before the first run (one time only)
1. **Join the Kaggle competition:** open https://www.kaggle.com/competitions/ieee-fraud-detection/rules and click **"I Understand and Accept"**. Without this step the download fails with a 403 error.
2. **Create a Kaggle API credential:** Kaggle → Settings → API → **Create New Token**.
3. Give the credential to Colab in **one** of these ways:
   - **(A) Colab Secrets (recommended):** click the 🔑 icon in the left sidebar and add either `KAGGLE_API_TOKEN` (new-style token) or `KAGGLE_USERNAME` + `KAGGLE_KEY` (values from `kaggle.json`). Turn **Notebook access** on.
   - **(B) Upload:** do nothing in advance. When Cell 3 runs, a **"Choose Files"** button appears; select your `kaggle.json`. It is saved to Google Drive and will not be requested again.
4. **Runtime → Change runtime type → T4 GPU.**
5. **Runtime → Run all.**

> Never hard-code a Kaggle token in this notebook or commit it to a repository.

## If the GPU or Colab session disconnects
Nothing is lost. Every step is saved to Google Drive (`MyDrive/AMAZTA`). Reconnect and choose **Run all**. Completed steps are skipped, and model training resumes from the last checkpoint (XGBoost every 100 rounds, the autoencoder every epoch).

**To start again from scratch**, delete everything inside the `AMAZTA` folder on Drive except `data_raw` (the data will not be downloaded again).

**Outputs:** `MyDrive/AMAZTA/results/` (CSV/JSON tables) and `MyDrive/AMAZTA/figures/` (300 dpi figures for the paper).


In [ ]:
# ===================== CELL 1: Setup, Google Drive, helpers =====================
import os, sys, json, time, gc, glob, zipfile, shutil, subprocess, warnings, pickle, re, random, itertools
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# QUICK_TEST = True only checks the code on a small synthetic dataset. Keep it False for real runs.
QUICK_TEST = os.environ.get('AMAZTA_QUICK_TEST', '0') == '1'
SEED = 42
random.seed(SEED); np.random.seed(SEED)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/AMAZTA'
else:
    BASE = os.path.abspath('./AMAZTA_work')
if QUICK_TEST:
    BASE += '_quicktest'

DIRS = {k: os.path.join(BASE, k) for k in ['data_raw', 'data', 'models', 'checkpoints', 'results', 'figures']}
for _d in DIRS.values():
    os.makedirs(_d, exist_ok=True)
RAW = DIRS['data_raw']
STATE_PATH = os.path.join(BASE, 'state.json')
LOG_PATH = os.path.join(BASE, 'run_log.txt')


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with open(LOG_PATH, 'a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass


def _json_default(o):
    if isinstance(o, np.ndarray):
        return o.tolist()
    if hasattr(o, 'item'):
        return o.item()
    return str(o)


def atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=_json_default)
    os.replace(tmp, path)


def read_json(path, default=None):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return default


def atomic_pickle(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f)
    os.replace(tmp, path)


def read_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


STATE = read_json(STATE_PATH, {}) or {}


def is_done(step):
    return bool(STATE.get(step, {}).get('done', False))


def mark_done(step, **info):
    STATE[step] = {'done': True, 'time': time.strftime('%Y-%m-%d %H:%M:%S'), **info}
    atomic_json(STATE, STATE_PATH)
    log(f'SAVED: {step} complete')


log(f'Working folder: {BASE}')
log(f'Steps already done: {[k for k in STATE if is_done(k)]}')

Mounted at /content/drive
[2026-10-01 10:48:53] Working folder: /content/drive/MyDrive/AMAZTA
[2026-10-01 10:48:54] Steps already done: []


In [ ]:
# ===================== CELL 2: Libraries + GPU check =====================
def ensure(mod, pip_name=None):
    try:
        return __import__(mod)
    except ImportError:
        if IN_COLAB:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or mod])
            try:
                return __import__(mod)
            except ImportError:
                pass
        return None


xgb = ensure('xgboost')
if xgb is not None and int(xgb.__version__.split('.')[0]) < 2:
    log('xgboost version is too old; using the scikit-learn fallback')
    xgb = None
torch = ensure('torch')
HAS_PARQUET = ensure('pyarrow') is not None

import matplotlib
if not IN_COLAB:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, accuracy_score, confusion_matrix, roc_curve, precision_recall_curve)

GPU = False
if torch is not None:
    GPU = bool(torch.cuda.is_available())
    torch.manual_seed(SEED)
    import torch.nn as nn

log(f"xgboost: {xgb.__version__ if xgb else 'NOT available (sklearn fallback)'} | "
    f"torch: {torch.__version__ if torch else 'NOT available (IsolationForest fallback)'} | GPU: {GPU}")
if IN_COLAB and not GPU:
    log('NOTE: no GPU found. The code also runs on CPU but more slowly. Runtime > Change runtime type > T4 GPU.')


def save_df(df, name):
    ext = '.parquet' if HAS_PARQUET else '.pkl'
    path = os.path.join(DIRS['data'], name + ext)
    tmp = path + '.tmp'
    if HAS_PARQUET:
        df.to_parquet(tmp, index=False)
    else:
        df.to_pickle(tmp)
    os.replace(tmp, path)
    return path


def load_df(name):
    p1 = os.path.join(DIRS['data'], name + '.parquet')
    p2 = os.path.join(DIRS['data'], name + '.pkl')
    if HAS_PARQUET and os.path.exists(p1):
        return pd.read_parquet(p1)
    if os.path.exists(p2):
        return pd.read_pickle(p2)
    raise FileNotFoundError(f'{name} not found. Please re-run the previous step.')

[2026-10-01 10:49:00] xgboost: 3.4.1 | torch: 2.11.0+cu128 | GPU: True


In [ ]:
# ===================== CELL 3: Auto download (Kaggle) =====================
# Kaggle API token (KGAT_...). Do NOT paste a token here; provide it through Colab Secrets
# (KAGGLE_API_TOKEN) or an environment variable instead.
KAGGLE_API_TOKEN = os.environ.get('KAGGLE_API_TOKEN', '')

COMP = 'ieee-fraud-detection'
NEEDED = ['train_transaction.csv', 'train_identity.csv']
ZIYA_DIR = os.path.join(RAW, 'ziya07')
os.makedirs(ZIYA_DIR, exist_ok=True)


class SetupError(Exception):
    pass


def raw_ready():
    return all(os.path.exists(os.path.join(RAW, f)) and os.path.getsize(os.path.join(RAW, f)) > 1000 for f in NEEDED)


def extract_zips(folder, wanted=None):
    for z in glob.glob(os.path.join(folder, '*.zip')):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    base = os.path.basename(m)
                    if not base or (wanted and base not in wanted):
                        continue
                    target = os.path.join(folder, base)
                    if os.path.exists(target) and os.path.getsize(target) > 0:
                        continue
                    tmp = target + '.tmp'
                    with zf.open(m) as src, open(tmp, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
                    os.replace(tmp, target)
                    log(f'Extracted {base}')
        except zipfile.BadZipFile:
            log(f'Deleted a corrupted zip file: {z}')
            os.remove(z)


_KAGGLE_UPGRADED = False


def setup_kaggle():
    """Configure Kaggle credentials from (1) env/secret token, (2) Colab Secrets, (3) kaggle.json on Drive, (4) upload."""
    global _KAGGLE_UPGRADED
    user = key = None
    token = (KAGGLE_API_TOKEN or '').strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            def _secret(name):
                try:
                    return userdata.get(name)
                except Exception:
                    return None
            token = token or (_secret('KAGGLE_API_TOKEN') or '').strip()
            user, key = _secret('KAGGLE_USERNAME'), _secret('KAGGLE_KEY')
        except Exception:
            pass
    if token:
        # Use the new-style token only; remove legacy credentials so they cannot take precedence
        for v in ['KAGGLE_USERNAME', 'KAGGLE_KEY']:
            os.environ.pop(v, None)
        kdir = os.path.expanduser('~/.kaggle'); os.makedirs(kdir, exist_ok=True)
        old_json = os.path.join(kdir, 'kaggle.json')
        if os.path.exists(old_json):
            os.remove(old_json)
        tp = os.path.join(kdir, 'access_token')
        with open(tp, 'w') as f:
            f.write(token)
        os.chmod(tp, 0o600)
        os.environ['KAGGLE_API_TOKEN'] = token
        if IN_COLAB and not _KAGGLE_UPGRADED:
            # New-style tokens require a recent version of the Kaggle client
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kaggle'])
            _KAGGLE_UPGRADED = True
        return True
    p = os.path.join(BASE, 'kaggle.json')
    if not (user and key) and os.path.exists(p):
        d = read_json(p, {}) or {}
        user, key = d.get('username'), d.get('key')
    if not (user and key) and IN_COLAB:
        # Upload button: select kaggle.json. It is saved to Drive and will not be requested again.
        from google.colab import files
        print('\n>>> Click the "Choose Files" button below and select your kaggle.json file <<<\n')
        up = files.upload()
        for fname, content in up.items():
            try:
                d = json.loads(content.decode('utf-8'))
                user, key = d.get('username'), d.get('key')
                if user and key:
                    atomic_json({'username': user, 'key': key}, p)
                    log(f'kaggle.json received and saved to Drive ({fname})')
                    break
            except Exception as e:
                log(f'{fname} does not look like a valid kaggle.json: {e}')
    if user and key:
        os.environ['KAGGLE_USERNAME'] = user
        os.environ['KAGGLE_KEY'] = key
        kdir = os.path.expanduser('~/.kaggle')
        os.makedirs(kdir, exist_ok=True)
        kp = os.path.join(kdir, 'kaggle.json')
        with open(kp, 'w') as f:
            json.dump({'username': user, 'key': key}, f)
        os.chmod(kp, 0o600)
    return bool(user and key)


def kaggle_cmd(args):
    exe = shutil.which('kaggle')
    if exe is None and IN_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
        exe = shutil.which('kaggle')
    if exe is None:
        return 1, 'kaggle CLI not found'
    r = subprocess.run([exe] + args, capture_output=True, text=True)
    return r.returncode, (r.stdout or '') + (r.stderr or '')


def make_fake_data():
    """QUICK_TEST only: small synthetic data with the IEEE-CIS schema."""
    rng = np.random.default_rng(SEED)
    n = 12000
    y = (rng.random(n) < 0.04).astype(int)
    tr = pd.DataFrame({'TransactionID': np.arange(2987000, 2987000 + n), 'isFraud': y,
                       'TransactionDT': np.sort(rng.integers(86400, 86400 * 180, n)),
                       'TransactionAmt': np.round(rng.lognormal(4, 1, n) * (1 + y), 2),
                       'ProductCD': rng.choice(list('WCRHS'), n)})
    for i in range(1, 7):
        tr[f'card{i}'] = rng.integers(100, 600, n).astype(float)
    tr['card4'] = rng.choice(['visa', 'mastercard', None], n)
    tr['card6'] = rng.choice(['debit', 'credit'], n)
    tr['addr1'] = rng.integers(100, 500, n).astype(float); tr['addr2'] = 87.0
    tr['dist1'] = np.where(rng.random(n) < 0.5, np.nan, rng.exponential(50, n) * (1 + 2 * y))
    tr['dist2'] = np.nan
    tr['P_emaildomain'] = rng.choice(['gmail.com', 'yahoo.com', None, 'anonymous.com'], n)
    tr['R_emaildomain'] = rng.choice(['gmail.com', None], n)
    for i in range(1, 15):
        tr[f'C{i}'] = rng.poisson(1 + 3 * y)
    for i in range(1, 16):
        tr[f'D{i}'] = np.where(rng.random(n) < 0.3, np.nan, rng.exponential(100, n) * (1 - 0.7 * y))
    for i in range(1, 10):
        tr[f'M{i}'] = rng.choice(['T', 'F', None], n)
    for i in range(1, 41):
        tr[f'V{i}'] = rng.normal(0, 1, n) + y * (i % 3)
    has_id = rng.random(n) < (0.25 + 0.4 * y)
    idf = pd.DataFrame({'TransactionID': tr['TransactionID'][has_id].values})
    m = len(idf)
    for i in range(1, 39):
        idf[f'id_{i:02d}'] = rng.normal(0, 1, m) if i < 12 else rng.choice(['Found', 'NotFound', 'New'], m)
    idf['DeviceType'] = rng.choice(['desktop', 'mobile'], m)
    idf['DeviceInfo'] = rng.choice(['Windows', 'iOS Device', 'MacOS', 'SM-G', 'rare1'], m)
    tr.to_csv(os.path.join(RAW, 'train_transaction.csv'), index=False)
    idf.to_csv(os.path.join(RAW, 'train_identity.csv'), index=False)
    k = 1000
    z = pd.DataFrame({'Transaction ID': [f'T{i}' for i in range(k)],
                      'Sender Account ID': [f'A{i}' for i in rng.integers(0, 300, k)],
                      'Receiver Account ID': [f'A{i}' for i in rng.integers(0, 300, k)],
                      'Transaction Amount': rng.lognormal(6, 1, k).round(2),
                      'Transaction Type': rng.choice(['Transfer', 'Withdrawal', 'Deposit'], k),
                      'Timestamp': pd.date_range('2025-01-01', periods=k, freq='h').astype(str),
                      'Transaction Status': rng.choice(['Success', 'Failed'], k),
                      'Fraud Flag': rng.choice([True, False], k),
                      'Geolocation (Latitude/Longitude)': [f'{rng.uniform(-90, 90):.4f} N, {rng.uniform(-180, 180):.4f} E' for _ in range(k)],
                      'Device Used': rng.choice(['Mobile', 'Desktop', 'Tablet'], k),
                      'Network Slice ID': rng.choice(['Slice1', 'Slice2', 'Slice3'], k),
                      'Latency (ms)': rng.integers(1, 100, k),
                      'Slice Bandwidth (Mbps)': rng.integers(50, 500, k),
                      'PIN Code': ['****'] * k})
    z.to_csv(os.path.join(ZIYA_DIR, 'fake_ziya.csv'), index=False)
    log('QUICK_TEST: synthetic data created')


def download_all():
    extract_zips(RAW, NEEDED)
    if raw_ready():
        log('IEEE-CIS data already present, skipping download')
    elif QUICK_TEST:
        make_fake_data()
    else:
        if not setup_kaggle():
            raise SetupError(
                'Kaggle credentials not found. Add KAGGLE_API_TOKEN to Colab Secrets, or re-run this cell and upload a valid kaggle.json.')
        for f in NEEDED:
            if os.path.exists(os.path.join(RAW, f)):
                continue
            log(f'Downloading {f} ... (the first run takes a few minutes)')
            code, out = kaggle_cmd(['competitions', 'download', '-c', COMP, '-f', f, '-p', RAW, '--force'])
            if code != 0:
                if '403' in out or 'Forbidden' in out or 'rules' in out.lower():
                    raise SetupError('Kaggle 403 (Forbidden). The credentials are valid, but this account has not joined the competition. '
                                     'With the same account, open https://www.kaggle.com/competitions/ieee-fraud-detection, '
                                     'click "Join Competition" / "Late Submission", accept the rules, then Run all.\n'
                                     f'Kaggle message: {out[-400:]}')
                if '401' in out or 'Unauthorized' in out:
                    raise SetupError('Kaggle 401: the token is invalid or expired. Create a new token in Kaggle Settings > API and '
                                     f'add it to Colab Secrets as KAGGLE_API_TOKEN.\nKaggle message: {out[-400:]}')
                raise SetupError(f'Download failed:\n{out[-800:]}')
            extract_zips(RAW, NEEDED)
        if not raw_ready():
            # sometimes the whole competition archive is delivered as one zip
            code, out = kaggle_cmd(['competitions', 'download', '-c', COMP, '-p', RAW])
            extract_zips(RAW, NEEDED)
        if not raw_ready():
            raise SetupError('Files could not be extracted. Please check the MyDrive/AMAZTA/data_raw folder.')
        log('IEEE-CIS download complete')

    # ziya07 (case study) - optional; the main pipeline continues even if this download fails
    if not glob.glob(os.path.join(ZIYA_DIR, '*.csv')) and not QUICK_TEST:
        try:
            setup_kaggle()
            code, out = kaggle_cmd(['datasets', 'download', '-d', 'ziya07/transaction-data-for-banking-operations',
                                    '-p', ZIYA_DIR, '--unzip'])
            extract_zips(ZIYA_DIR)
            log('ziya07 dataset downloaded' if glob.glob(os.path.join(ZIYA_DIR, '*.csv'))
                else f'ziya07 download failed (optional): {out[-300:]}')
        except Exception as e:
            log(f'ziya07 download skip (optional): {e}')


download_all()

[2026-10-01 10:49:04] Downloading train_transaction.csv ... (the first run takes a few minutes)
[2026-10-01 10:49:15] Extracted train_transaction.csv
[2026-10-01 10:49:15] Downloading train_identity.csv ... (the first run takes a few minutes)
[2026-10-01 10:49:17] Extracted train_identity.csv
[2026-10-01 10:49:17] IEEE-CIS download complete
[2026-10-01 10:49:18] ziya07 dataset downloaded


In [ ]:
# ===================== CELL 4: STEP 1, Load + merge + encode =====================
_DF = None


def get_df():
    global _DF
    if _DF is None:
        _DF = load_df('merged')
        log(f'Merged data loaded: {_DF.shape}')
    return _DF


def step1_load_merge():
    if is_done('step1_merge'):
        log('SKIP step1 (already done)')
        return
    log('Reading CSVs ...')
    tr = pd.read_csv(os.path.join(RAW, 'train_transaction.csv'))
    idf = pd.read_csv(os.path.join(RAW, 'train_identity.csv'))
    idf.columns = [c.replace('-', '_') for c in idf.columns]
    idf['has_identity'] = 1.0
    df = tr.merge(idf, on='TransactionID', how='left')
    del tr, idf; gc.collect()
    df['has_identity'] = df['has_identity'].fillna(0.0)
    df['hour'] = ((df['TransactionDT'] // 3600) % 24).astype('float32')
    df['dow'] = ((df['TransactionDT'] // 86400) % 7).astype('float32')
    encoders = {}
    for c in list(df.columns):
        if not pd.api.types.is_numeric_dtype(df[c]):
            codes, uniques = pd.factorize(df[c].astype('object'), sort=True)
            codes = codes.astype('float32')
            codes[codes < 0] = np.nan
            df[c] = codes
            encoders[c] = [str(u) for u in uniques]
        elif df[c].dtype == 'float64':
            df[c] = df[c].astype('float32')
        elif df[c].dtype == 'bool':
            df[c] = df[c].astype('float32')
    df['isFraud'] = df['isFraud'].astype('int8')
    df = df.reset_index(drop=True)
    atomic_pickle(encoders, os.path.join(DIRS['data'], 'encoders.pkl'))
    save_df(df, 'merged')
    mark_done('step1_merge', rows=int(len(df)), cols=int(df.shape[1]), fraud_rate=float(df['isFraud'].mean()))
    global _DF
    _DF = df


step1_load_merge()

[2026-10-01 10:49:18] Reading CSVs ...
[2026-10-01 10:50:04] SAVED: step1_merge complete


In [ ]:
# ===================== CELL 5: STEP 2, Time-based split + agent feature groups =====================
def step2_split():
    if is_done('step2_split'):
        log('SKIP step2 (already done)')
        return
    df = get_df()
    order = np.argsort(df['TransactionDT'].values, kind='stable')
    n = len(order); a = int(n * 0.70); b = int(n * 0.85)
    np.savez(os.path.join(DIRS['data'], 'splits.npz'), train=order[:a], val=order[a:b], test=order[b:])
    cols = list(df.columns)
    pat = lambda p: [c for c in cols if re.fullmatch(p, c)]
    device = [c for c in ['DeviceType', 'DeviceInfo', 'id_30', 'id_31', 'id_32', 'id_33', 'has_identity'] if c in cols]
    identity = [c for c in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2',
                            'P_emaildomain', 'R_emaildomain'] if c in cols] + [c for c in pat(r'id_\d+') if c not in device]
    behavior = [c for c in ['TransactionAmt'] if c in cols] + pat(r'C\d+') + pat(r'D\d+') + pat(r'M\d+') + pat(r'V\d+')
    context = [c for c in ['ProductCD', 'dist1', 'dist2', 'hour', 'dow'] if c in cols]
    allf = [c for c in cols if c not in ('TransactionID', 'isFraud', 'TransactionDT')]
    groups = dict(identity=identity, device=device, behavior=behavior, context=context, all=allf)
    atomic_json(groups, os.path.join(DIRS['data'], 'feature_groups.json'))
    mark_done('step2_split', train=int(a), val=int(b - a), test=int(n - b),
              n_features={k: len(v) for k, v in groups.items()})


def get_splits():
    s = np.load(os.path.join(DIRS['data'], 'splits.npz'))
    return {k: s[k] for k in ['train', 'val', 'test']}


def get_groups():
    return read_json(os.path.join(DIRS['data'], 'feature_groups.json'))


def mat(df, idx, cols):
    return df[cols].iloc[idx].to_numpy(dtype=np.float32, na_value=np.nan)


step2_split()
log(f"Split sizes: {STATE['step2_split']}")

[2026-10-01 10:50:04] SAVED: step2_split complete
[2026-10-01 10:50:04] Split sizes: {'done': True, 'time': '2026-10-01 10:50:04', 'train': 413378, 'val': 88581, 'test': 88581, 'n_features': {'identity': 44, 'device': 7, 'behavior': 378, 'context': 5, 'all': 434}}


In [ ]:
# ===================== CELL 6: STEP 3, Tree agents (Identity, Device, Threat Detection) =====================
PATIENCE = 3
CHUNK = 20 if QUICK_TEST else 100


def train_tree_agent(name, cols, max_rounds, params_extra=None):
    step = f'agent_{name}'
    if is_done(step):
        log(f'SKIP {step} (already trained)')
        return
    df = get_df(); sp = get_splits()
    y = df['isFraud'].values
    Xtr, ytr = mat(df, sp['train'], cols), y[sp['train']]
    Xva, yva = mat(df, sp['val'], cols), y[sp['val']]
    meta_p = os.path.join(DIRS['checkpoints'], f'{name}_meta.json')
    meta = read_json(meta_p, None) or {'best_auc': -1.0, 'best_round': 0, 'bad': 0, 'history': []}
    if xgb is not None:
        params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
                  'device': 'cuda' if GPU else 'cpu', 'eta': 0.05, 'max_depth': 6, 'subsample': 0.8,
                  'colsample_bytree': 0.6, 'min_child_weight': 5, 'seed': SEED, 'verbosity': 0}
        params.update(params_extra or {})
        dtr = xgb.DMatrix(Xtr, label=ytr, missing=np.nan, feature_names=cols)
        dva = xgb.DMatrix(Xva, missing=np.nan, feature_names=cols)
        ckpt = os.path.join(DIRS['checkpoints'], f'{name}_xgb.json')
        booster = None
        if os.path.exists(ckpt):
            try:
                booster = xgb.Booster(); booster.load_model(ckpt)
                log(f'RESUME {name}: {booster.num_boosted_rounds()} rounds already trained')
            except Exception as e:
                log(f'Checkpoint could not be loaded, restarting: {e}'); booster = None
                meta = {'best_auc': -1.0, 'best_round': 0, 'bad': 0, 'history': []}
        while True:
            done_r = booster.num_boosted_rounds() if booster is not None else 0
            if done_r >= max_rounds or meta['bad'] >= PATIENCE:
                break
            booster = xgb.train(params, dtr, num_boost_round=CHUNK, xgb_model=booster)
            r = booster.num_boosted_rounds()
            auc = float(roc_auc_score(yva, booster.predict(dva)))
            meta['history'].append([r, auc])
            if auc > meta['best_auc'] + 1e-4:
                meta.update(best_auc=auc, best_round=r, bad=0)
            else:
                meta['bad'] += 1
            tmp = ckpt.replace('.json', '_tmp.json')
            booster.save_model(tmp); os.replace(tmp, ckpt)
            atomic_json(meta, meta_p)
            log(f'{name}: rounds={r} val_AUC={auc:.4f} (best {meta["best_auc"]:.4f} @ {meta["best_round"]}) [saved]')
        final = os.path.join(DIRS['models'], f'{name}_xgb.json')
        shutil.copy(ckpt, final)
        mark_done(step, kind='xgb', model=final, best_round=int(meta['best_round']),
                  val_auc=float(meta['best_auc']), history=meta['history'])
        del dtr, dva
    else:
        from sklearn.ensemble import HistGradientBoostingClassifier
        ck = os.path.join(DIRS['checkpoints'], f'{name}_hgb.pkl')
        model = read_pickle(ck) if os.path.exists(ck) else None
        if model is not None:
            log(f'RESUME {name}: {model.max_iter} iterations already trained')
        while True:
            it = model.max_iter if model is not None else 0
            if it >= max_rounds or meta['bad'] >= PATIENCE:
                break
            if model is None:
                model = HistGradientBoostingClassifier(max_iter=CHUNK, learning_rate=0.1, warm_start=True,
                                                       early_stopping=False, random_state=SEED)
            else:
                model.set_params(max_iter=it + CHUNK)
            model.fit(Xtr, ytr)
            auc = float(roc_auc_score(yva, model.predict_proba(Xva)[:, 1]))
            meta['history'].append([model.max_iter, auc])
            if auc > meta['best_auc'] + 1e-4:
                meta.update(best_auc=auc, best_round=model.max_iter, bad=0)
            else:
                meta['bad'] += 1
            atomic_pickle(model, ck); atomic_json(meta, meta_p)
            log(f'{name}: iters={model.max_iter} val_AUC={auc:.4f} [saved]')
        final = os.path.join(DIRS['models'], f'{name}_hgb.pkl')
        shutil.copy(ck, final)
        mark_done(step, kind='hgb', model=final, best_round=int(meta['best_round']),
                  val_auc=float(meta['best_auc']), history=meta['history'])
    del Xtr, Xva; gc.collect()


G = get_groups()
MAXR = 100 if QUICK_TEST else 2000
train_tree_agent('identity', G['identity'], max_rounds=MAXR // 2)
train_tree_agent('device', G['device'], max_rounds=MAXR // 4, params_extra={'max_depth': 4})
train_tree_agent('threat', G['all'], max_rounds=MAXR, params_extra={'max_depth': 8, 'colsample_bytree': 0.5})

[2026-10-01 10:50:05] identity: rounds=100 val_AUC=0.7869 (best 0.7869 @ 100) [saved]
[2026-10-01 10:50:07] identity: rounds=200 val_AUC=0.7964 (best 0.7964 @ 200) [saved]
[2026-10-01 10:50:10] identity: rounds=300 val_AUC=0.8029 (best 0.8029 @ 300) [saved]
[2026-10-01 10:50:13] identity: rounds=400 val_AUC=0.8079 (best 0.8079 @ 400) [saved]
[2026-10-01 10:50:20] identity: rounds=500 val_AUC=0.8126 (best 0.8126 @ 500) [saved]
[2026-10-01 10:50:26] identity: rounds=600 val_AUC=0.8143 (best 0.8143 @ 600) [saved]
[2026-10-01 10:50:34] identity: rounds=700 val_AUC=0.8162 (best 0.8162 @ 700) [saved]
[2026-10-01 10:50:41] identity: rounds=800 val_AUC=0.8167 (best 0.8167 @ 800) [saved]
[2026-10-01 10:50:52] identity: rounds=900 val_AUC=0.8174 (best 0.8174 @ 900) [saved]
[2026-10-01 10:51:02] identity: rounds=1000 val_AUC=0.8175 (best 0.8174 @ 900) [saved]
[2026-10-01 10:51:02] SAVED: agent_identity complete
[2026-10-01 10:51:03] device: rounds=100 val_AUC=0.6806 (best 0.6806 @ 100) [saved]
[2

In [ ]:
# ===================== CELL 7: STEP 4, Behavioral Monitoring Agent (Autoencoder) =====================
AE_EPOCHS = 2 if QUICK_TEST else 12
AE_BATCH = 4096

if torch is not None:
    class AE(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 16))
            self.dec = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, d))

        def forward(self, x):
            return self.dec(self.enc(x))


def torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def beh_transform(X, s):
    X = np.where(np.isnan(X), s['med'], X)
    X = (X - s['mean']) / s['std']
    return np.clip(X, -5, 5).astype(np.float32)


def step4_behavior():
    if is_done('agent_behavior'):
        log('SKIP agent_behavior (already trained)')
        return
    df = get_df(); sp = get_splits(); cols = get_groups()['behavior']
    y = df['isFraud'].values
    tr_norm = sp['train'][y[sp['train']] == 0]
    sc_path = os.path.join(DIRS['models'], 'behavior_scaler.npz')
    if os.path.exists(sc_path):
        s = dict(np.load(sc_path))
    else:
        X = mat(df, tr_norm, cols)
        med = np.nanmedian(X, axis=0); med = np.where(np.isnan(med), 0, med).astype(np.float32)
        Xf = np.where(np.isnan(X), med, X)
        mean = Xf.mean(0).astype(np.float32); std = Xf.std(0).astype(np.float32); std[std < 1e-6] = 1.0
        s = {'med': med, 'mean': mean, 'std': std}
        np.savez(sc_path, **s); del X, Xf
    Xn = beh_transform(mat(df, tr_norm, cols), s)

    if torch is not None:
        dev = 'cuda' if GPU else 'cpu'
        model = AE(Xn.shape[1]).to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        ck = os.path.join(DIRS['checkpoints'], 'behavior_ae_ckpt.pt')
        start, losses = 0, []
        if os.path.exists(ck):
            try:
                c = torch_load(ck, dev)
                model.load_state_dict(c['model']); opt.load_state_dict(c['opt'])
                start, losses = int(c['epoch']), list(c['losses'])
                log(f'RESUME autoencoder from epoch {start}')
            except Exception as e:
                log(f'AE checkpoint could not be loaded, restarting: {e}')
        Xt = torch.from_numpy(Xn)
        n = Xt.shape[0]
        for ep in range(start, AE_EPOCHS):
            model.train()
            g = torch.Generator().manual_seed(SEED + ep)
            perm = torch.randperm(n, generator=g)
            tot = 0.0
            for i in range(0, n, AE_BATCH):
                xb = Xt[perm[i:i + AE_BATCH]].to(dev)
                loss = ((model(xb) - xb) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
                tot += float(loss.item()) * xb.shape[0]
            losses.append(tot / n)
            tmp = ck + '.tmp'
            torch.save({'epoch': ep + 1, 'model': model.state_dict(), 'opt': opt.state_dict(), 'losses': losses}, tmp)
            os.replace(tmp, ck)
            log(f'Autoencoder epoch {ep + 1}/{AE_EPOCHS} loss={losses[-1]:.4f} [saved]')
        final = os.path.join(DIRS['models'], 'behavior_ae.pt')
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, final + '.tmp')
        os.replace(final + '.tmp', final)
        mark_done('agent_behavior', kind='ae', model=final, dim=int(Xn.shape[1]), losses=losses)
    else:
        from sklearn.ensemble import IsolationForest
        rng = np.random.default_rng(SEED)
        sub = Xn[rng.choice(len(Xn), min(len(Xn), 100000), replace=False)]
        iso = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(sub)
        final = os.path.join(DIRS['models'], 'behavior_iso.pkl')
        atomic_pickle(iso, final)
        mark_done('agent_behavior', kind='iso', model=final, dim=int(Xn.shape[1]))
    del Xn; gc.collect()


step4_behavior()

[2026-10-01 10:57:43] Autoencoder epoch 1/12 loss=0.2441 [saved]
[2026-10-01 10:57:45] Autoencoder epoch 2/12 loss=0.1213 [saved]
[2026-10-01 10:57:47] Autoencoder epoch 3/12 loss=0.0956 [saved]
[2026-10-01 10:57:49] Autoencoder epoch 4/12 loss=0.0844 [saved]
[2026-10-01 10:57:52] Autoencoder epoch 5/12 loss=0.0771 [saved]
[2026-10-01 10:57:55] Autoencoder epoch 6/12 loss=0.0717 [saved]
[2026-10-01 10:57:57] Autoencoder epoch 7/12 loss=0.0676 [saved]
[2026-10-01 10:57:59] Autoencoder epoch 8/12 loss=0.0644 [saved]
[2026-10-01 10:58:01] Autoencoder epoch 9/12 loss=0.0621 [saved]
[2026-10-01 10:58:04] Autoencoder epoch 10/12 loss=0.0600 [saved]
[2026-10-01 10:58:07] Autoencoder epoch 11/12 loss=0.0584 [saved]
[2026-10-01 10:58:09] Autoencoder epoch 12/12 loss=0.0569 [saved]
[2026-10-01 10:58:09] SAVED: agent_behavior complete


In [ ]:
# ===================== CELL 8: STEP 5, Context Agent + Vulnerability (rules learned from history) =====================
def te_fit(v, y, smoothing=50.0):
    prior = float(y.mean())
    s = pd.DataFrame({'v': v, 'y': y}).groupby('v')['y'].agg(['sum', 'count'])
    enc = (s['sum'] + prior * smoothing) / (s['count'] + smoothing)
    return {float(k): float(val) for k, val in enc.items()}, prior


def ctx_keys(df_part, edges):
    out = {}
    for c in ['ProductCD', 'hour', 'dow']:
        if c in df_part.columns:
            out[c] = np.nan_to_num(df_part[c].to_numpy(dtype=np.float64, na_value=np.nan), nan=-999.0)
    for c in ['dist1', 'dist2']:
        if c in df_part.columns and c in edges:
            v = df_part[c].to_numpy(dtype=np.float64, na_value=np.nan)
            b = np.digitize(v, edges[c]).astype(np.float64)
            b[np.isnan(v)] = -1.0
            out[c] = b
    return out


def step5_context():
    if is_done('agent_context'):
        log('SKIP agent_context (already done)')
        return
    df = get_df(); sp = get_splits()
    tr = df.iloc[sp['train']]
    y = tr['isFraud'].values
    edges = {}
    for c in ['dist1', 'dist2']:
        if c in tr.columns:
            v = tr[c].dropna().values
            if len(v) > 100:
                edges[c] = np.unique(np.quantile(v, np.linspace(0.1, 0.9, 9))).tolist()
    keys = ctx_keys(tr, edges)
    maps = {}
    prior = float(y.mean())
    for c, v in keys.items():
        maps[c], prior = te_fit(v, y)
    atomic_pickle({'edges': edges, 'maps': maps, 'prior': prior}, os.path.join(DIRS['models'], 'context_agent.pkl'))
    mark_done('agent_context', features=list(maps.keys()))


step5_context()

[2026-10-01 10:58:09] SAVED: agent_context complete


In [ ]:
# ===================== CELL 9: AMAZTA Engine (all agents together) =====================
def to_rank(x, ref_sorted):
    return np.searchsorted(ref_sorted, x, side='right') / float(len(ref_sorted))


class TreeAgent:
    def __init__(self, name, cols):
        info = STATE[f'agent_{name}']
        self.cols, self.kind = cols, info['kind']
        if self.kind == 'xgb':
            self.b = xgb.Booster(); self.b.load_model(info['model'])
            self.b.set_param({'device': 'cpu', 'verbosity': 0})
            self.best = max(1, int(info['best_round']))
        else:
            self.m = read_pickle(info['model'])

    def predict(self, X):
        if self.kind == 'xgb':
            d = xgb.DMatrix(X, missing=np.nan, feature_names=self.cols)
            return self.b.predict(d, iteration_range=(0, self.best))
        return self.m.predict_proba(X)[:, 1]


class BehaviorAgent:
    def __init__(self, cols):
        info = STATE['agent_behavior']
        self.cols, self.kind = cols, info['kind']
        self.s = dict(np.load(os.path.join(DIRS['models'], 'behavior_scaler.npz')))
        if self.kind == 'ae':
            self.model = AE(int(info['dim']))
            self.model.load_state_dict(torch_load(info['model'], 'cpu'))
            self.model.eval()
        else:
            self.iso = read_pickle(info['model'])

    def score(self, X):
        Z = beh_transform(X, self.s)
        if self.kind == 'ae':
            out = []
            with torch.no_grad():
                for i in range(0, len(Z), 65536):
                    zb = torch.from_numpy(Z[i:i + 65536])
                    out.append(((self.model(zb) - zb) ** 2).mean(1).numpy())
            return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)
        return -self.iso.score_samples(Z)


class ContextAgent:
    def __init__(self):
        self.a = read_pickle(os.path.join(DIRS['models'], 'context_agent.pkl'))

    def score(self, df_part):
        keys = ctx_keys(df_part, self.a['edges'])
        parts = []
        for c, m in self.a['maps'].items():
            if c in keys:
                parts.append(pd.Series(keys[c]).map(m).fillna(self.a['prior']).to_numpy())
        return np.mean(parts, axis=0) if parts else np.full(len(df_part), self.a['prior'])


def vulnerability(df_part):
    parts = []
    if 'has_identity' in df_part: parts.append(1.0 - df_part['has_identity'].to_numpy(dtype=np.float64))
    for c in ['DeviceInfo', 'P_emaildomain', 'card2']:
        if c in df_part: parts.append(df_part[c].isna().to_numpy().astype(np.float64))
    return np.mean(parts, axis=0) if parts else np.zeros(len(df_part))


def entity_key(df_part):
    c1 = df_part['card1'].astype(str) if 'card1' in df_part else pd.Series('na', index=df_part.index)
    a1 = df_part['addr1'].astype(str) if 'addr1' in df_part else pd.Series('na', index=df_part.index)
    return (c1 + '_' + a1).to_numpy()


class AMAZTAEngine:
    """Identity, Device, Behavioral, Threat, Context agents + Risk, Policy, Response agents."""

    def __init__(self):
        g = get_groups()
        self.g = g
        self.identity = TreeAgent('identity', g['identity'])
        self.device = TreeAgent('device', g['device'])
        self.threat = TreeAgent('threat', g['all'])
        self.behavior = BehaviorAgent(g['behavior'])
        self.context = ContextAgent()
        rp = os.path.join(DIRS['models'], 'norm_refs.npz')
        self.refs = dict(np.load(rp)) if os.path.exists(rp) else None
        self.fusion = read_json(os.path.join(DIRS['results'], 'fusion.json'), None)
        self.entity_trust = {}   # dynamic trust memory (shared context of the Coordination Agent)

    def raw(self, dp):
        f = lambda cols: dp[cols].to_numpy(dtype=np.float32, na_value=np.nan)
        return {'p_id': self.identity.predict(f(self.g['identity'])),
                'p_dev': self.device.predict(f(self.g['device'])),
                'p_thr': self.threat.predict(f(self.g['all'])),
                'beh': self.behavior.score(f(self.g['behavior'])),
                'ctx': self.context.score(dp),
                'V': vulnerability(dp)}

    def evidence(self, r):
        R = self.refs
        rk = {k: to_rank(r[k], R[k]) for k in ['p_id', 'p_dev', 'p_thr', 'beh', 'ctx']}
        return {'I': 1 - rk['p_id'], 'D': 1 - rk['p_dev'], 'B': 1 - rk['beh'], 'C': 1 - rk['ctx'],
                'A': rk['p_thr'], 'Ba': rk['beh'], 'Cp': rk['ctx'], 'V': np.asarray(r['V'], dtype=np.float64)}

    def decide(self, dp):
        """Full pipeline: evidence -> trust -> dynamic trust -> risk -> action."""
        fu = self.fusion
        e = self.evidence(self.raw(dp))
        wt, wr = fu['wT'], fu['wR']
        T = wt['I'] * e['I'] + wt['D'] * e['D'] + wt['B'] * e['B'] + wt['C'] * e['C']
        a = fu['alpha']
        ents = entity_key(dp)
        Tdyn = np.empty_like(T)
        for i, (k, t) in enumerate(zip(ents, T)):
            prev = self.entity_trust.get(k)
            v = t if prev is None else a * prev + (1 - a) * t
            self.entity_trust[k] = v
            Tdyn[i] = v
        R = (wr['T'] * (1 - Tdyn) + wr['A'] * e['A'] + wr['V'] * e['V'] + wr['Ba'] * e['Ba'] + wr['Cp'] * e['Cp'])
        return T, Tdyn, R, policy_actions(R, Tdyn, fu['thr'])


ACTIONS = ['Allow', 'Restrict', 'Deny', 'Isolate']


def policy_actions(R, T, thr):
    act = np.zeros(len(R), dtype=int)
    act[(R >= thr['theta_R']) | (T < thr['theta_T'])] = 1
    deny = R >= thr['theta_D']
    act[deny] = 2
    act[deny & (T < thr['theta_I'])] = 3
    return act

In [ ]:
# ===================== CELL 10: STEP 6, Agent outputs (validation + test) =====================
def step6_score():
    if is_done('step6_score'):
        log('SKIP step6 (already scored)')
        return
    df = get_df(); sp = get_splits()
    eng = AMAZTAEngine()
    parts_dir = os.path.join(DIRS['checkpoints'], 'score_parts'); os.makedirs(parts_dir, exist_ok=True)
    idx_all = np.concatenate([sp['val'], sp['test']])
    split_lbl = np.array(['val'] * len(sp['val']) + ['test'] * len(sp['test']))
    CH = 50000
    for j, i in enumerate(range(0, len(idx_all), CH)):
        pp = os.path.join(parts_dir, f'part_{j:04d}.pkl')
        if os.path.exists(pp):
            continue
        idx = idx_all[i:i + CH]
        dp = df.iloc[idx]
        r = eng.raw(dp)
        out = pd.DataFrame({k: np.asarray(v, dtype=np.float64) for k, v in r.items()})
        out['row'] = idx; out['split'] = split_lbl[i:i + CH]
        out['isFraud'] = dp['isFraud'].values; out['TransactionDT'] = dp['TransactionDT'].values
        out['entity'] = entity_key(dp)
        out['TransactionAmt'] = dp['TransactionAmt'].values if 'TransactionAmt' in dp else 0.0
        out.to_pickle(pp + '.tmp'); os.replace(pp + '.tmp', pp)
        log(f'Scored chunk {j} ({i + len(idx)}/{len(idx_all)}) [saved]')
    raw = pd.concat([pd.read_pickle(p) for p in sorted(glob.glob(os.path.join(parts_dir, 'part_*.pkl')))],
                    ignore_index=True)
    v = raw[raw['split'] == 'val']
    refs = {k: np.sort(v[k].values) for k in ['p_id', 'p_dev', 'p_thr', 'beh', 'ctx']}
    np.savez(os.path.join(DIRS['models'], 'norm_refs.npz'), **refs)
    eng.refs = refs
    ev = eng.evidence({k: raw[k].values for k in ['p_id', 'p_dev', 'p_thr', 'beh', 'ctx', 'V']})
    for k, val in ev.items():
        raw[k] = val
    save_df(raw, 'agent_outputs')
    mark_done('step6_score', rows=int(len(raw)))


step6_score()

[2026-10-01 10:58:15] Scored chunk 0 (50000/177162) [saved]
[2026-10-01 10:58:22] Scored chunk 1 (100000/177162) [saved]
[2026-10-01 10:58:27] Scored chunk 2 (150000/177162) [saved]
[2026-10-01 10:58:30] Scored chunk 3 (177162/177162) [saved]
[2026-10-01 10:58:30] SAVED: step6_score complete


In [ ]:
# ===================== CELL 11: STEP 7, Trust/Risk fusion + Dynamic trust + Policy thresholds =====================
def simplex(k, step=0.1, min_w=0.1):
    """All weight combinations that sum to 1. min_w ensures every agent keeps a minimum share."""
    n = int(round(1 / step)); m = int(round(min_w / step))
    return [np.array(c) / n for c in itertools.product(range(m, n + 1), repeat=k) if sum(c) == n]


def dyn_trust(ao, alpha):
    """T(t+1) = alpha*T(t) + (1-alpha)*E(t), per entity, in time order (validation + test)."""
    if alpha == 0:
        return ao['T_static'].values
    o = ao.sort_values('TransactionDT', kind='stable')
    t = o.groupby('entity', sort=False)['T_static'].ewm(alpha=1 - alpha, adjust=False).mean()
    t = t.reset_index(level=0, drop=True)
    return t.reindex(ao.index).values


def risk(ao, T, wr):
    return (wr['T'] * (1 - T) + wr['A'] * ao['A'].values + wr['V'] * ao['V'].values
            + wr['Ba'] * ao['Ba'].values + wr['Cp'] * ao['Cp'].values)


def best_f1_threshold(y, s):
    qs = np.unique(np.quantile(s, np.linspace(0.80, 0.999, 120)))
    f = [f1_score(y, s >= q, zero_division=0) for q in qs]
    return float(qs[int(np.argmax(f))])


def step7_fusion():
    if is_done('step7_fusion'):
        log('SKIP step7 (already done)')
        return
    ao = load_df('agent_outputs')
    val = ao['split'].values == 'val'
    yv = ao['isFraud'].values[val]
    # ---- Trust weights (wI, wD, wB, wC), sum = 1 ----
    E = ao[['I', 'D', 'B', 'C']].values
    best = (-1, None)
    for w in simplex(4):
        ap = average_precision_score(yv, 1 - E[val] @ w)
        if ap > best[0]: best = (ap, w)
    wT = dict(zip(['I', 'D', 'B', 'C'], best[1].tolist()))
    ao['T_static'] = E @ best[1]
    log(f'Trust weights: {wT} (val AP of 1-T = {best[0]:.4f})')
    # ---- Risk weights (wT, wA, wV, wB, wC'), sum = 1 ----
    M = np.column_stack([1 - ao['T_static'].values, ao['A'].values, ao['V'].values, ao['Ba'].values, ao['Cp'].values])
    best = (-1, None)
    for w in simplex(5):
        ap = average_precision_score(yv, M[val] @ w)
        if ap > best[0]: best = (ap, w)
    wR = dict(zip(['T', 'A', 'V', 'Ba', 'Cp'], best[1].tolist()))
    log(f'Risk weights: {wR} (val AP = {best[0]:.4f})')
    # ---- Dynamic trust persistence factor alpha ----
    alpha_res = []
    for a in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        Td = dyn_trust(ao, a)
        alpha_res.append([a, float(average_precision_score(yv, risk(ao, Td, wR)[val]))])
    alpha = max(alpha_res, key=lambda x: x[1])[0]
    log(f'alpha search (alpha, val AP): {alpha_res} -> best alpha={alpha}')
    ao['T_dyn'] = dyn_trust(ao, alpha)
    ao['R'] = risk(ao, ao['T_dyn'].values, wR)
    ao['R_static'] = risk(ao, ao['T_static'].values, wR)
    # ---- Policy thresholds (from the validation set) ----
    Rv, Tv, normal = ao['R'].values[val], ao['T_dyn'].values[val], yv == 0
    theta_D = best_f1_threshold(yv, Rv)
    theta_R = min(float(np.quantile(Rv[normal], 0.95)), theta_D * 0.999)
    thr = {'theta_D': theta_D, 'theta_R': theta_R,
           'theta_T': float(np.quantile(Tv[normal], 0.05)), 'theta_I': float(np.quantile(Tv[normal], 0.02))}
    thr_static = {**thr, 'theta_D': best_f1_threshold(yv, ao['R_static'].values[val])}
    ao['action'] = policy_actions(ao['R'].values, ao['T_dyn'].values, thr)
    fusion = {'wT': wT, 'wR': wR, 'alpha': alpha, 'alpha_search': alpha_res, 'thr': thr, 'thr_static': thr_static}
    atomic_json(fusion, os.path.join(DIRS['results'], 'fusion.json'))
    save_df(ao, 'fused_outputs')
    mark_done('step7_fusion', **fusion)


step7_fusion()

[2026-10-01 10:58:33] Trust weights: {'I': 0.6, 'D': 0.1, 'B': 0.2, 'C': 0.1} (val AP of 1-T = 0.3470)
[2026-10-01 10:58:36] Risk weights: {'T': 0.1, 'A': 0.6, 'V': 0.1, 'Ba': 0.1, 'Cp': 0.1} (val AP = 0.4939)
[2026-10-01 10:58:39] alpha search (alpha, val AP): [[0.0, 0.49389766404640556], [0.2, 0.49191285613366825], [0.4, 0.4901783569200354], [0.5, 0.48949018448341763], [0.6, 0.48923663959858266], [0.7, 0.4891690580895254], [0.8, 0.4889619497358975], [0.9, 0.48737534187672754]] -> best alpha=0.0
[2026-10-01 10:58:41] SAVED: step7_fusion complete


In [ ]:
# ===================== CELL 12: STEP 8, Evaluation + Baselines + Ablation =====================
def metrics(y, pred, score, name):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'Model': name, 'Accuracy': accuracy_score(y, pred), 'Precision': precision_score(y, pred, zero_division=0),
            'Recall': recall_score(y, pred, zero_division=0), 'F1': f1_score(y, pred, zero_division=0),
            'FPR': fp / max(fp + tn, 1), 'ROC_AUC': roc_auc_score(y, score), 'PR_AUC': average_precision_score(y, score)}


def step8_eval():
    if is_done('step8_eval'):
        log('SKIP step8 (already done)')
        return
    ao = load_df('fused_outputs'); fu = read_json(os.path.join(DIRS['results'], 'fusion.json'))
    val, test = ao['split'].values == 'val', ao['split'].values == 'test'
    yv, yt = ao['isFraud'].values[val], ao['isFraud'].values[test]
    rows = []
    # Baseline 1: Static rule-based Zero Trust (no ML)
    amt = ao['TransactionAmt'].values
    amt_hi = np.quantile(amt[val], 0.90)
    s_rule = ao['V'].values + (amt >= amt_hi)
    rows.append(metrics(yt, ((ao['V'].values >= 0.75) & (amt >= amt_hi))[test], s_rule[test], 'Static rule-based ZT (no ML)'))
    # Baseline 2: Centralized single ML model (Threat Detection model only)
    p = ao['p_thr'].values
    th = best_f1_threshold(yv, p[val])
    rows.append(metrics(yt, p[test] >= th, p[test], 'Centralized single ML (XGBoost)'))
    # AMAZTA static trust
    Rs = ao['R_static'].values
    rows.append(metrics(yt, Rs[test] >= fu['thr_static']['theta_D'], Rs[test], 'AMAZTA (static trust)'))
    # AMAZTA adaptive (proposed)
    R = ao['R'].values
    rows.append(metrics(yt, ao['action'].values[test] >= 2, R[test], f"AMAZTA adaptive (alpha={fu['alpha']})"))
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(DIRS['results'], 'main_results.csv'), index=False)
    # Action distribution (test)
    act = ao['action'].values[test]
    dist = pd.DataFrame({'Action': ACTIONS,
                         'Normal': [int(((act == i) & (yt == 0)).sum()) for i in range(4)],
                         'Fraud': [int(((act == i) & (yt == 1)).sum()) for i in range(4)]})
    dist.to_csv(os.path.join(DIRS['results'], 'action_distribution.csv'), index=False)
    # Ablation: remove one agent at a time
    wT, wR = fu['wT'], fu['wR']
    abl = []
    def run(wt, wr):
        st = sum(wt.values()); sr = sum(wr.values())
        if sr == 0: return np.nan, np.nan
        wr = {k: v / sr for k, v in wr.items()}
        if st == 0:
            T = np.full(len(ao), 0.5)
        else:
            wt = {k: v / st for k, v in wt.items()}
            T = sum(wt[k] * ao[k].values for k in wt)
        r = risk(ao, T, wr)[test]
        return roc_auc_score(yt, r), average_precision_score(yt, r)
    auc0, ap0 = run(dict(wT), dict(wR))
    abl.append({'Removed agent': 'None (full AMAZTA)', 'ROC_AUC': auc0, 'PR_AUC': ap0})
    for name, kt, kr in [('Identity Agent', 'I', None), ('Device Trust Agent', 'D', None),
                         ('Behavioral Agent', 'B', 'Ba'), ('Context Agent', 'C', 'Cp'),
                         ('Threat Detection Agent', None, 'A'), ('Vulnerability check', None, 'V')]:
        wt, wr = dict(wT), dict(wR)
        if kt: wt[kt] = 0.0
        if kr: wr[kr] = 0.0
        a, b = run(wt, wr)
        abl.append({'Removed agent': name, 'ROC_AUC': a, 'PR_AUC': b})
    pd.DataFrame(abl).to_csv(os.path.join(DIRS['results'], 'ablation.csv'), index=False)
    mark_done('step8_eval')


step8_eval()
print(pd.read_csv(os.path.join(DIRS['results'], 'main_results.csv')).round(4).to_string(index=False))
print()
print(pd.read_csv(os.path.join(DIRS['results'], 'action_distribution.csv')).to_string(index=False))
print()
print(pd.read_csv(os.path.join(DIRS['results'], 'ablation.csv')).round(4).to_string(index=False))

[2026-10-01 10:58:43] SAVED: step8_eval complete
                          Model  Accuracy  Precision  Recall     F1    FPR  ROC_AUC  PR_AUC
   Static rule-based ZT (no ML)    0.9527     0.0313  0.0120 0.0173 0.0134   0.3734  0.0307
Centralized single ML (XGBoost)    0.9696     0.5757  0.4862 0.5272 0.0129   0.8975  0.5444
          AMAZTA (static trust)    0.9603     0.4368  0.4865 0.4603 0.0226   0.8979  0.3991
    AMAZTA adaptive (alpha=0.0)    0.9603     0.4368  0.4865 0.4603 0.0226   0.8979  0.3991

  Action  Normal  Fraud
   Allow   76180    946
Restrict    7384    637
    Deny    1228    418
 Isolate     706   1082

         Removed agent  ROC_AUC  PR_AUC
    None (full AMAZTA)   0.8979  0.3991
        Identity Agent   0.8960  0.3774
    Device Trust Agent   0.8981  0.3997
      Behavioral Agent   0.8877  0.3385
         Context Agent   0.8934  0.3226
Threat Detection Agent   0.7868  0.1253
   Vulnerability check   0.9008  0.4656


In [ ]:
# ===================== CELL 13: STEP 9, Response latency + Scalability =====================
def step9_latency():
    if is_done('step9_latency'):
        log('SKIP step9 (already done)')
        return
    df = get_df(); sp = get_splits(); test_idx = sp['test']
    eng = AMAZTAEngine()
    rng = np.random.default_rng(SEED)
    single = []
    for i in rng.choice(test_idx, size=min(200, len(test_idx)), replace=False):
        dp = df.iloc[[i]]
        t0 = time.perf_counter(); eng.decide(dp); single.append((time.perf_counter() - t0) * 1000)
    lat = {'single_mean_ms': float(np.mean(single)), 'single_p95_ms': float(np.percentile(single, 95)),
           'single_median_ms': float(np.median(single))}
    scal = []
    for bs in [100, 1000, 10000, 50000]:
        if bs > len(test_idx): break
        dp = df.iloc[test_idx[:bs]]
        eng.entity_trust = {}
        t0 = time.perf_counter(); eng.decide(dp); dt = time.perf_counter() - t0
        scal.append({'requests': bs, 'total_sec': dt, 'ms_per_request': dt * 1000 / bs, 'requests_per_sec': bs / dt})
        log(f'Batch {bs}: {dt:.2f}s ({bs / dt:.0f} req/s)')
    pd.DataFrame(scal).to_csv(os.path.join(DIRS['results'], 'scalability.csv'), index=False)
    atomic_json(lat, os.path.join(DIRS['results'], 'latency.json'))
    mark_done('step9_latency', **lat)


step9_latency()
print(read_json(os.path.join(DIRS['results'], 'latency.json')))
print(pd.read_csv(os.path.join(DIRS['results'], 'scalability.csv')).round(4).to_string(index=False))

[2026-10-01 10:58:49] Batch 100: 0.10s (998 req/s)
[2026-10-01 10:58:49] Batch 1000: 0.16s (6293 req/s)
[2026-10-01 10:58:50] Batch 10000: 1.11s (8994 req/s)
[2026-10-01 10:58:55] Batch 50000: 4.74s (10555 req/s)
[2026-10-01 10:58:55] SAVED: step9_latency complete
{'single_mean_ms': 21.560988599997017, 'single_p95_ms': 63.498988600002804, 'single_median_ms': 13.145084000029783}
 requests  total_sec  ms_per_request  requests_per_sec
      100     0.1002          1.0018          998.1889
     1000     0.1589          0.1589         6292.7066
    10000     1.1119          0.1112         8993.5621
    50000     4.7373          0.0947        10554.5657


In [ ]:
# ===================== CELL 14: STEP 10, Figures for the paper (300 dpi) =====================
def savefig(name):
    p = os.path.join(DIRS['figures'], name)
    plt.tight_layout(); plt.savefig(p, dpi=300); plt.close()
    log(f'Figure saved: {name}')


def step10_figures():
    ao = load_df('fused_outputs'); fu = read_json(os.path.join(DIRS['results'], 'fusion.json'))
    test = ao['split'].values == 'test'; yt = ao['isFraud'].values[test]
    scores = {'Centralized ML': ao['p_thr'].values[test], 'AMAZTA static': ao['R_static'].values[test],
              'AMAZTA adaptive': ao['R'].values[test]}
    jobs = []

    def roc():
        plt.figure(figsize=(5, 4.5))
        for k, s in scores.items():
            f, t, _ = roc_curve(yt, s); plt.plot(f, t, label=f'{k} (AUC={roc_auc_score(yt, s):.3f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=0.8); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title('ROC Curve (test)'); plt.legend(fontsize=8); savefig('fig_roc.png')
    jobs.append(roc)

    def pr():
        plt.figure(figsize=(5, 4.5))
        for k, s in scores.items():
            p, r, _ = precision_recall_curve(yt, s); plt.plot(r, p, label=f'{k} (AP={average_precision_score(yt, s):.3f})')
        plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve (test)')
        plt.legend(fontsize=8); savefig('fig_pr.png')
    jobs.append(pr)

    def cm():
        c = confusion_matrix(yt, ao['action'].values[test] >= 2, labels=[0, 1])
        plt.figure(figsize=(4, 3.6)); plt.imshow(c, cmap='Blues')
        for i in range(2):
            for j in range(2):
                plt.text(j, i, f'{c[i, j]:,}', ha='center', va='center',
                         color='white' if c[i, j] > c.max() / 2 else 'black')
        plt.xticks([0, 1], ['Allowed', 'Blocked']); plt.yticks([0, 1], ['Normal', 'Fraud'])
        plt.title('AMAZTA Confusion Matrix'); savefig('fig_confusion.png')
    jobs.append(cm)

    def actions():
        d = pd.read_csv(os.path.join(DIRS['results'], 'action_distribution.csv'))
        x = np.arange(4); fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
        for a, col in zip(ax, ['Normal', 'Fraud']):
            v = d[col].values / max(d[col].sum(), 1) * 100
            a.bar(x, v, color=['#2e7d32', '#f9a825', '#c62828', '#4a148c'])
            a.set_xticks(x); a.set_xticklabels(d['Action']); a.set_ylabel('% of requests'); a.set_title(col)
        savefig('fig_actions.png')
    jobs.append(actions)

    def trust_hist():
        plt.figure(figsize=(5, 4))
        T = ao['T_dyn'].values[test]
        plt.hist(T[yt == 0], bins=50, alpha=0.6, density=True, label='Normal')
        plt.hist(T[yt == 1], bins=50, alpha=0.6, density=True, label='Fraud')
        plt.xlabel('Dynamic trust score T'); plt.ylabel('Density'); plt.title('Trust distribution'); plt.legend()
        savefig('fig_trust_distribution.png')
    jobs.append(trust_hist)

    def alpha_plot():
        a = np.array(fu['alpha_search'])
        plt.figure(figsize=(5, 3.5)); plt.plot(a[:, 0], a[:, 1], 'o-')
        plt.xlabel('Trust persistence factor alpha'); plt.ylabel('Validation PR-AUC'); plt.title('Sensitivity to alpha')
        savefig('fig_alpha_sensitivity.png')
    jobs.append(alpha_plot)

    def ablation():
        d = pd.read_csv(os.path.join(DIRS['results'], 'ablation.csv'))
        plt.figure(figsize=(6, 3.8)); plt.barh(d['Removed agent'], d['PR_AUC'])
        plt.xlabel('Test PR-AUC'); plt.title('Ablation: agent removed'); plt.gca().invert_yaxis()
        savefig('fig_ablation.png')
    jobs.append(ablation)

    def scal():
        d = pd.read_csv(os.path.join(DIRS['results'], 'scalability.csv'))
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
        ax[0].plot(d['requests'], d['requests_per_sec'], 'o-'); ax[0].set_xscale('log')
        ax[0].set_xlabel('Concurrent requests'); ax[0].set_ylabel('Throughput (req/s)')
        ax[1].plot(d['requests'], d['ms_per_request'], 'o-', color='C1'); ax[1].set_xscale('log')
        ax[1].set_xlabel('Concurrent requests'); ax[1].set_ylabel('Latency per request (ms)')
        savefig('fig_scalability.png')
    jobs.append(scal)

    def importance():
        if STATE['agent_threat']['kind'] != 'xgb': return
        b = xgb.Booster(); b.load_model(STATE['agent_threat']['model'])
        imp = pd.Series(b.get_score(importance_type='gain')).sort_values(ascending=False).head(20)
        plt.figure(figsize=(6, 5)); plt.barh(imp.index[::-1], imp.values[::-1])
        plt.xlabel('Gain'); plt.title('Threat Detection Agent: top 20 features'); savefig('fig_feature_importance.png')
    jobs.append(importance)

    def training():
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
        for n in ['identity', 'device', 'threat']:
            h = np.array(STATE[f'agent_{n}'].get('history', []))
            if len(h): ax[0].plot(h[:, 0], h[:, 1], 'o-', label=n)
        ax[0].set_xlabel('Boosting rounds'); ax[0].set_ylabel('Validation AUC'); ax[0].legend()
        l = STATE['agent_behavior'].get('losses', [])
        if l: ax[1].plot(range(1, len(l) + 1), l, 'o-')
        ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Autoencoder MSE')
        savefig('fig_training_curves.png')
    jobs.append(training)

    for j in jobs:
        try:
            j()
        except Exception as e:
            plt.close('all'); log(f'Figure {j.__name__} skipped: {e}')
    mark_done('step10_figures')


step10_figures()

[2026-10-01 10:58:56] Figure saved: fig_roc.png
[2026-10-01 10:58:56] Figure saved: fig_pr.png
[2026-10-01 10:58:56] Figure saved: fig_confusion.png
[2026-10-01 10:58:57] Figure saved: fig_actions.png
[2026-10-01 10:58:57] Figure saved: fig_trust_distribution.png
[2026-10-01 10:58:57] Figure saved: fig_alpha_sensitivity.png
[2026-10-01 10:58:58] Figure saved: fig_ablation.png
[2026-10-01 10:58:58] Figure saved: fig_scalability.png
[2026-10-01 10:58:59] Figure saved: fig_feature_importance.png
[2026-10-01 10:59:00] Figure saved: fig_training_curves.png
[2026-10-01 10:59:00] SAVED: step10_figures complete


In [ ]:
# ===================== CELL 15: STEP 11, Case study on ziya07 banking data (optional) =====================
def find_col(df, *keys):
    for c in df.columns:
        lc = c.lower()
        if all(k in lc for k in keys):
            return c
    return None


def rank01(x):
    return pd.Series(x).rank(pct=True).fillna(0.5).values


def step11_case_study():
    if is_done('step11_case_study'):
        log('SKIP step11 (already done)')
        return
    files = glob.glob(os.path.join(ZIYA_DIR, '*.csv'))
    if not files:
        log('ziya07 data not found, skipping the case study (optional)')
        return
    try:
        z = pd.read_csv(files[0])
        fc = find_col(z, 'fraud')
        if fc is None:
            log('Fraud column not found, skipping the case study'); return
        yv = z[fc]
        y = (yv.astype(float) > 0).astype(int).values if pd.api.types.is_numeric_dtype(yv) or yv.dtype == bool \
            else yv.astype(str).str.lower().isin(['1', 'true', 'yes', 'fraud', 'fraudulent']).astype(int).values
        amt = find_col(z, 'amount'); dev = find_col(z, 'device'); lat = find_col(z, 'latency')
        bw = find_col(z, 'bandwidth'); snd = find_col(z, 'sender'); st = find_col(z, 'status')
        typ = find_col(z, 'type'); sl = find_col(z, 'network', 'slice')
        n = len(z)
        num = lambda c: pd.to_numeric(z[c], errors='coerce').fillna(0).values if c else np.zeros(n)
        # Identity agent: new or rarely used account = lower trust
        I = rank01(z[snd].map(z[snd].value_counts()).values) if snd else np.full(n, 0.5)
        # Device agent: rare device = lower trust
        D = rank01(z[dev].map(z[dev].value_counts()).values) if dev else np.full(n, 0.5)
        # Context (network) agent: higher latency + lower bandwidth = higher contextual risk
        Cp = (rank01(num(lat)) + (1 - rank01(num(bw)))) / 2
        # Behavioral agent: IsolationForest
        from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
        from sklearn.model_selection import cross_val_predict, StratifiedKFold
        Xb = np.column_stack([num(amt), num(lat), num(bw)])
        Ba = rank01(-IsolationForest(random_state=SEED).fit(Xb).score_samples(Xb))
        # Threat agent: 5-fold out-of-fold ML
        X = pd.DataFrame({'amt': num(amt), 'lat': num(lat), 'bw': num(bw)})
        for c in [dev, typ, sl, st]:
            if c: X[c] = pd.factorize(z[c])[0]
        if len(np.unique(y)) == 2 and min(np.bincount(y)) >= 5:
            A = cross_val_predict(HistGradientBoostingClassifier(max_iter=100, random_state=SEED), X, y,
                                  cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
            A = rank01(A)
        else:
            A = np.full(n, 0.5)
        V = (z[st].astype(str).str.lower().str.contains('fail')).astype(float).values if st else np.zeros(n)
        fu = read_json(os.path.join(DIRS['results'], 'fusion.json'), None)
        wT = fu['wT'] if fu else {'I': .25, 'D': .25, 'B': .25, 'C': .25}
        wR = fu['wR'] if fu else {'T': .2, 'A': .2, 'V': .2, 'Ba': .2, 'Cp': .2}
        T = wT['I'] * I + wT['D'] * D + wT['B'] * (1 - Ba) + wT['C'] * (1 - Cp)
        R = wR['T'] * (1 - T) + wR['A'] * A + wR['V'] * V + wR['Ba'] * Ba + wR['Cp'] * Cp
        thr = {'theta_D': float(np.quantile(R, 0.90)), 'theta_R': float(np.quantile(R, 0.70)),
               'theta_T': float(np.quantile(T, 0.05)), 'theta_I': float(np.quantile(T, 0.02))}
        act = policy_actions(R, T, thr)
        out = z.copy(); out['Trust'] = T; out['Risk'] = R; out['Action'] = [ACTIONS[a] for a in act]
        out.to_csv(os.path.join(DIRS['results'], 'case_study_ziya07_decisions.csv'), index=False)
        summ = {'rows': n, 'fraud_rate': float(y.mean()),
                'ROC_AUC': float(roc_auc_score(y, R)) if len(np.unique(y)) == 2 else None,
                'PR_AUC': float(average_precision_score(y, R)) if len(np.unique(y)) == 2 else None,
                'actions': {ACTIONS[i]: int((act == i).sum()) for i in range(4)}}
        atomic_json(summ, os.path.join(DIRS['results'], 'case_study_ziya07.json'))
        plt.figure(figsize=(5, 3.5)); plt.bar(ACTIONS, [summ['actions'][a] for a in ACTIONS],
                                              color=['#2e7d32', '#f9a825', '#c62828', '#4a148c'])
        plt.ylabel('Transactions'); plt.title('Case study: AMAZTA decisions (ziya07)'); savefig('fig_case_study_actions.png')
        log(f'Case study: {summ}')
        log('NOTE: ziya07 is synthetic data. Do not report its AUC as a main result; use it only to demonstrate the decision workflow.')
        mark_done('step11_case_study', **summ)
    except Exception as e:
        log(f'Case study failed and was skipped (main results are unaffected): {e}')


step11_case_study()

[2026-10-01 10:59:03] Figure saved: fig_case_study_actions.png
[2026-10-01 10:59:03] Case study: {'rows': 1000, 'fraud_rate': 0.481, 'ROC_AUC': 0.5218695796730479, 'PR_AUC': 0.4941284000687061, 'actions': {'Allow': 675, 'Restrict': 225, 'Deny': 95, 'Isolate': 5}}
[2026-10-01 10:59:03] NOTE: ziya07 is synthetic data. Do not report its AUC as a main result; use it only to demonstrate the decision workflow.
[2026-10-01 10:59:03] SAVED: step11_case_study complete


In [ ]:
# ===================== CELL 16: Final summary =====================
log('================ ALL DONE ================')
print('\nMAIN RESULTS (test set):')
print(pd.read_csv(os.path.join(DIRS['results'], 'main_results.csv')).round(4).to_string(index=False))
fu = read_json(os.path.join(DIRS['results'], 'fusion.json'))
print('\nLearned trust weights:', {k: round(v, 2) for k, v in fu['wT'].items()})
print('Learned risk weights :', {k: round(v, 2) for k, v in fu['wR'].items()})
print('Best alpha:', fu['alpha'])
print('\nLatency:', read_json(os.path.join(DIRS['results'], 'latency.json')))
print(f'\nAll files are saved in: {BASE}')
for d in ['results', 'figures']:
    print(f'  {d}/:', sorted(os.listdir(DIRS[d])))

[2026-10-01 10:59:03] ================ ALL DONE ================

MAIN RESULTS (test set):
                          Model  Accuracy  Precision  Recall     F1    FPR  ROC_AUC  PR_AUC
   Static rule-based ZT (no ML)    0.9527     0.0313  0.0120 0.0173 0.0134   0.3734  0.0307
Centralized single ML (XGBoost)    0.9696     0.5757  0.4862 0.5272 0.0129   0.8975  0.5444
          AMAZTA (static trust)    0.9603     0.4368  0.4865 0.4603 0.0226   0.8979  0.3991
    AMAZTA adaptive (alpha=0.0)    0.9603     0.4368  0.4865 0.4603 0.0226   0.8979  0.3991

Learned trust weights: {'I': 0.6, 'D': 0.1, 'B': 0.2, 'C': 0.1}
Learned risk weights : {'T': 0.1, 'A': 0.6, 'V': 0.1, 'Ba': 0.1, 'Cp': 0.1}
Best alpha: 0.0

Latency: {'single_mean_ms': 21.560988599997017, 'single_p95_ms': 63.498988600002804, 'single_median_ms': 13.145084000029783}

All files are saved in: /content/drive/MyDrive/AMAZTA
  results/: ['ablation.csv', 'action_distribution.csv', 'case_study_ziya07.json', 'case_study_ziya07_decisions.

In [ ]:
# ===================== CELL 17: Download all results as one zip file =====================
import os, zipfile, time
from google.colab import files

stamp = time.strftime('%Y%m%d_%H%M')
zip_name = f'AMAZTA_results_{stamp}.zip'
zip_path = os.path.join('/content', zip_name)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['results', 'figures']:
        for f in sorted(os.listdir(DIRS[folder])):
            zf.write(os.path.join(DIRS[folder], f), arcname=f'{folder}/{f}')
    for f in ['state.json', 'run_log.txt']:
        p = os.path.join(BASE, f)
        if os.path.exists(p):
            zf.write(p, arcname=f)
    for f in ['feature_groups.json']:
        p = os.path.join(DIRS['data'], f)
        if os.path.exists(p):
            zf.write(p, arcname=f'data/{f}')

shutil.copy(zip_path, os.path.join(BASE, zip_name))   # also keep a copy on Drive
print(f'Zip ready: {zip_name} ({os.path.getsize(zip_path) / 1e6:.1f} MB)')
files.download(zip_path)

Zip ready: AMAZTA_results_20261001_1103.zip (0.9 MB)


In [1]:
# ===================== CELL 18: Software and hardware environment =====================
import sys, platform, xgboost, torch, sklearn, pandas, numpy
print('Python      :', sys.version.split()[0])
print('XGBoost     :', xgboost.__version__)
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.version.cuda)
print('scikit-learn:', sklearn.__version__)
print('pandas      :', pandas.__version__)
print('NumPy       :', numpy.__version__)
print('OS          :', platform.platform())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | head -2

Python      : 3.13.15
XGBoost     : 3.4.1
PyTorch     : 2.11.0+cu130
CUDA        : 13.0
scikit-learn: 1.6.1
pandas      : 2.2.3
NumPy       : 2.1.3
OS          : Linux-6.6.122+-x86_64-with-glibc2.39
Tesla T4, 15360 MiB
               total        used        free      shared  buff/cache   available
Mem:              12           1           6           0           4          11
